# Weak Strata Deep-Dive (1B)

**Purpose**: examine the weak categories identified in 01a in more depth. Covers case-mix-adjusted
annotator effects, interactions between categories, individual failing frames for manual review,
and a combined directional screen across all listed factors.

**Model**: U-Net + ResNet-34 encoder, baseline configuration (no cascade, no compound loss, no
targeted augmentation), same run as 01a.

**Input**: per-frame results CSV exported by the baseline training notebook, joined with the
annotator metadata spreadsheet. Uses the same `baseline_common.py` module as 01a, so both
notebooks build `frame_df` identically.

## Scope

This notebook takes the weak categories 01a found and asks: **are these effects real once you
control for other factors, and where specifically should manually review focus?**

It covers:
1. Video-level performance, so repeated frames from the same video don't dominate a comparison
2. Case-mix-adjusted annotator weakness. Is a raw annotator effect real, or does it disappear
   once you account for which surgery types/sizes that annotator happened to get assigned?
3. Interaction screens (e.g. size x a challenge, thinness x size, annotator x size), plus a challenge x size
   failure-rate heatmap. The highest-yield failure surfaces are often combinations, not single factors
4. A per-frame failure catalogue for manual review, with identifiers kept alongside failure tags
5. A combined directional effect screen across all listed factors at once
6. Summary figures: mean IoU by frame-level tool size band and by thin/wide band, IoU under
   smoke, blood, motion blur
   and exposure, and a size-conditional degradation panel that shows whether weak conditions hit
   every size band or mainly tiny tools
7. A gallery of the 20 lowest-IoU frames next to their ground truth and prediction masks

## Out of scope (see Notebook 01a)

Overall baseline performance and the one-factor category breakdowns are not repeated here, see
**01a** for those.

## 1. Install dependencies
Run this cell to install the packages required for this notebook, pinned in `requirements.txt`.

In [ ]:
from pathlib import Path

_req = Path("requirements.txt")
if not _req.exists():
	_req = Path("01_baseline_analysis") / "requirements.txt"
if not _req.exists():
	raise FileNotFoundError(f"requirements.txt not found from cwd={Path.cwd()}")
get_ipython().run_line_magic("pip", f"install -q -r {_req.as_posix()}")


## 2. Setup and configuration
Imports the shared helpers, then sets the file paths and output directory for this notebook.

In [ ]:
import sys
from pathlib import Path

# Kernel cwd is often the repo root; put this folder on sys.path so baseline_common imports.
_NB_DIR = Path.cwd()
if not (_NB_DIR / "baseline_common.py").exists():
	_NB_DIR = _NB_DIR / "01_baseline_analysis"
if not (_NB_DIR / "baseline_common.py").exists():
	raise ModuleNotFoundError(f"baseline_common.py not found from cwd={Path.cwd()}")
sys.path.insert(0, str(_NB_DIR.resolve()))

import importlib
import matplotlib.pyplot as plt
import baseline_common
importlib.reload(baseline_common)  # pick up edits to baseline_common without a kernel restart
from baseline_common import *

# ── Data sources ──
# 00_model_training writes: gs://{CKPT_BUCKET}/experiments/{RUN_NAME}/{RUN_ID}/
GCS_BUCKET  = "laparoscopic-seg-REPLACE_K_NUMBER"
CKPT_BUCKET = "training-checkpoints-REPLACE_K_NUMBER"
RUN_NAME    = "baseline-seed42-clean_cv_Unet_resnet34_bce_lr1e-04_p20"
RUN_ID      = "20260813-232105"
GCS_RUN_DIR = f"gs://{CKPT_BUCKET}/experiments/{RUN_NAME}/{RUN_ID}"
FRAMES_GCS  = f"gs://{GCS_BUCKET}/trainingFrames"
SHEET_ID    = "REPLACE_SHEET_ID"

CACHE_DIR  = Path("/tmp/laparoscopic_strata_cache")
OUTPUT_DIR = Path("/tmp/laparoscopic_strata_1b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH     = fetch_per_image_results(GCS_RUN_DIR, CACHE_DIR)
ANNOTATIONS_PATH = fetch_annotations(SHEET_ID, CACHE_DIR)

# ── Frame files for the worst-case gallery (section 13) ──
FRAME_DATA_ROOT = fetch_frame_data(FRAMES_GCS, CACHE_DIR)
IMAGES_DIR      = FRAME_DATA_ROOT / "JPEGImages"
MASKS_DIR       = FRAME_DATA_ROOT / "SegmentationClass"
PRED_MASKS_ROOT = fetch_predicted_masks(GCS_RUN_DIR, CACHE_DIR)

# ── Interaction screens ──
MIN_INTERACTION_FRAMES = 10	# interaction cells smaller than this are excluded from ranked breakdowns

print("GCS run:", GCS_RUN_DIR)
print("Results:", RESULTS_PATH)
print("Annotations:", ANNOTATIONS_PATH)
print("Frames:", IMAGES_DIR)
print("Pred masks:", PRED_MASKS_ROOT)
print("Output dir:", OUTPUT_DIR)


## 3. Helper functions
1B-specific helpers not covered by baseline_common: video-level lookups, interaction/pivot table builders, and the figure output settings used by sections 12 and 13.

In [ ]:
from typing import cast

from matplotlib.axes import Axes
from matplotlib.ticker import PercentFormatter

VISUAL_DIR = OUTPUT_DIR / "figures"
VISUAL_DIR.mkdir(parents=True, exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

COLORS = {
    "risk": PRIMARY,
    "volume": SECONDARY,
    "good": ACCENT,
    "warning": ACCENT_WARM,
    "gray": RULE,
}


def first_nonnull(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan


def mode_or_first(series):
    values = series.dropna()
    if len(values) == 0:
        return np.nan
    return values.value_counts(dropna=False).index[0]


def interaction_summary(df, left, right, min_rows=MIN_INTERACTION_FRAMES):
    out = summarize_groups(df, [left, right], min_rows=min_rows, min_videos=1)
    return out.sort_values(["failure_any_rate", "iou_mean", "n_frames"], ascending=[False, True, False])


def pivot_metric(df, index, columns, value="failure_any", min_rows=MIN_INTERACTION_FRAMES, agg="mean"):
    # cells below min_rows are blanked, so sparse combinations don't get read as signal
    count = df.pivot_table(index=index, columns=columns, values="key", aggfunc="size", fill_value=0)
    metric = df.pivot_table(index=index, columns=columns, values=value, aggfunc=agg)
    return count, metric.where(count >= min_rows)


def save_and_show(fig, filename):
    path = VISUAL_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved {path}")
    if "agg" not in plt.get_backend().lower():
        plt.show()
    plt.close(fig)
    return path


def pretty_size_label(value):
    # "1_tiny" -> "Tiny", for chart tick labels
    return str(value).split("_")[-1].title()


def plot_heatmap_from_table(df, index, columns, value, title, filename, min_rows=MIN_INTERACTION_FRAMES):
    # blank cells under min_rows via pivot_metric, same rule as the ranked interaction tables
    count, metric = pivot_metric(df, index, columns, value=value, min_rows=min_rows)
    if metric.empty:
        print(f"No data for {title}")
        return None
    metric = metric.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if metric.empty:
        print(f"No cells pass min_rows for {title}")
        return None
    counts = count.reindex(index=metric.index, columns=metric.columns)
    values = metric.values.astype(float)
    finite = np.isfinite(values)
    if not finite.any():
        print(f"No finite values for {title}")
        return None
    vmax = 1.0 if value.startswith("failure") or value == "failure_any" else float(np.nanmax(values))
    n_rows, n_cols = values.shape
    data = np.ma.masked_invalid(values)

    # rc_context + pcolormesh: whitegrid's ax.grid lines land on imshow tick centres and
    # strike through the cell labels; cell edges here sit on the borders instead.
    with plt.rc_context({"axes.grid": False}):
        fig, ax = plt.subplots(figsize=(max(8, 0.85 * n_cols), max(5, 0.45 * n_rows)))
        im = ax.pcolormesh(
            np.arange(n_cols + 1),
            np.arange(n_rows + 1),
            data,
            cmap=SEQUENTIAL_CMAP,
            vmin=0,
            vmax=vmax,
            edgecolors="white",
            linewidths=0.8,
        )
        ax.invert_yaxis()
        ax.set_xticks(np.arange(n_cols) + 0.5)
        col_labels = [
            pretty_size_label(c) if str(c).startswith(("0_", "1_", "2_", "3_", "4_")) else str(c)
            for c in metric.columns
        ]
        ax.set_xticklabels(col_labels, rotation=45, ha="right")
        ax.set_yticks(np.arange(n_rows) + 0.5)
        ax.set_yticklabels(metric.index.astype(str))
        ax.tick_params(which="both", length=0)
        ax.grid(visible=False, which="both", axis="both")
        ax.set_title(title)
        for row_i in range(n_rows):
            for col_i in range(n_cols):
                val = values[row_i, col_i]
                if np.isfinite(val):
                    n_value = counts.iloc[row_i, col_i]
                    n_label = int(n_value) if pd.notna(n_value) else 0
                    value_label = f"{val:.0%}" if 0 <= val <= 1 else f"{val:.2f}"
                    color = "white" if val > vmax * 0.55 else RULE
                    ax.text(
                        col_i + 0.5,
                        row_i + 0.5,
                        f"{value_label}\nn={n_label}",
                        ha="center",
                        va="center",
                        color=color,
                        fontsize=7,
                    )
        cbar = fig.colorbar(im, ax=ax)
        if value.startswith("failure") or value == "failure_any":
            cbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
            cbar.ax.grid(False)
        cbar.set_label(value)
    return save_and_show(fig, filename)


def ax_at(grid, row_index, column_index) -> Axes:
    # subplots() hands back a plain array, so the cast keeps the type checker quiet
    return cast(Axes, grid[row_index, column_index])

## 4. Load and pair data
Load the results CSV and annotation spreadsheet and join them on a frame key.

In [ ]:
frame_df, metadata_mismatches, invalid_annotations = load_and_pair(RESULTS_PATH, ANNOTATIONS_PATH)

display(metadata_mismatches)
display(frame_df[["key", "image", *META_COLS, "Status"]].head())

## 5. Derived metadata, instrument size and thinness classes
Compute the failure flags and frame-level size bands using the same area cuts as the training
notebook (tiny < 4.5%, small < 8%, medium < 19%, large >= 19%), plus the thin/wide band the run
was scored under: the run's own `thin_class`, cut at the 25th percentile of mean local width
(`thin_cut_px`, 22.81 px at 512 x 512).

In [ ]:
# size_cls already uses the training cuts (via add_derived_columns); mirror into size_band for 01b names.
SIZE_BAND_LABELS = SIZE_LABELS


def add_size_bands(df):
    """Copy frame-level size_cls into size_band columns used by the rest of 01b.
    Cuts: tiny < 4.5%, small < 8%, medium < 19%, large >= 19%."""
    df = df.copy()
    if "size_cls" not in df.columns:
        raise KeyError("size_cls missing; run add_derived_columns before add_size_bands")
    df["size_band"] = df["size_cls"]
    df["size_band_label"] = df["size_cls_label"]

    # 01a screens whole challenge combinations, 1B needs the individual labels too
    df["challenge_primary"] = df["challenge_tokens"].map(lambda xs: xs[0] if xs else "None")
    df["challenge_count"] = df["challenge_tokens"].map(len)
    return df


frame_df = add_derived_columns(frame_df)
frame_df, cutoffs, failure_thresholds = add_failure_flags(frame_df)
frame_df = add_size_bands(frame_df)

display(cutoffs)

# band boundaries, so the tiny band can be described in the write-up
band_overview = frame_df.groupby("size_band_label").agg(
    frames=("key", "size"),
    videos=("Video ID", "nunique"),
    area_pct_min=("tool_area_pct", "min"),
    area_pct_max=("tool_area_pct", "max"),
    iou_mean=("iou", "mean"),
).reset_index()
display(band_overview)

# the same, for thinness. width_px min and max bracket the stored thin_cut_px.
thin_overview = frame_df.groupby("thin_cls_label").agg(
    frames=("key", "size"),
    videos=("Video ID", "nunique"),
    width_px_min=("width_px", "min"),
    width_px_max=("width_px", "max"),
    iou_mean=("iou", "mean"),
).reset_index()
display(thin_overview)

## 6. Video-level performance
Aggregate frame-level results to one row per video, so videos with more sampled frames don't dominate a comparison.

In [ ]:
video_df = frame_df.groupby("Video ID", dropna=False).agg(
    key=("key", "first"),
    image=("image", first_nonnull),
    frame_count=("key", "size"),
    iou=("iou", "mean"),
    dice=("dice", "mean"),
    precision=("precision", "mean"),
    hd95=("hd95", "mean"),
    msd=("msd", "mean"),
    asd=("asd", "mean"),
    tool_pixels=("tool_pixels", "mean"),
    fp_pixels=("fp_pixels", "mean"),
    tool_area_pct=("tool_area_pct", "mean"),
    fp_area_pct=("fp_area_pct", "mean"),
    pred_fp_share=("pred_fp_share", "mean"),
    Annotator=("Annotator", mode_or_first),
    **{
        "Surgery Type": ("Surgery Type", mode_or_first),
        "Video Resolution": ("Video Resolution", mode_or_first),
        "Robotic": ("Robotic", mode_or_first),
        "No Tools": ("No Tools", mode_or_first),
    },
    Challenges=("Challenges", mode_or_first),
    challenge_primary=("challenge_primary", mode_or_first),
    challenge_combo=("challenge_combo", mode_or_first),
    size_cls_label=("size_cls_label", mode_or_first),
    size_band_label=("size_band_label", mode_or_first),
    thin_cls_label=("thin_cls_label", mode_or_first),
    failure_any=("failure_any", "max"),
    failure_rate=("failure_any", "mean"),
    failure_score=("failure_score", "mean"),
    failure_severe_iou=("failure_severe_iou", "max"),
    failure_weak_iou=("failure_weak_iou", "max"),
    failure_boundary_outlier=("failure_boundary_outlier", "max"),
    failure_low_precision=("failure_low_precision", "max"),
    failure_fp_outlier=("failure_fp_outlier", "max"),
    failure_no_tool_fp=("failure_no_tool_fp", "max"),
).reset_index()

for col in ["failure_any", *FAILURE_FLAG_COLS]:
    video_df[col] = video_df[col].astype(bool)

print(f"Video-level rows: {len(video_df):,}")
display(video_df[["Video ID", "frame_count", "iou", "hd95", "fp_pixels", "Annotator", "size_band_label", "failure_rate"]].head())

# same screens as 01a but one row per video, as a sanity check against repeated frames.
# n_frames counts videos here, not frames.
video_screen = video_df.copy()
video_screen["Surgery Type Top"] = collapse_rare(video_screen["Surgery Type"], top_n=25, min_count=MIN_GROUP_VIDEOS)

video_size_table = summarize_groups(video_screen, "size_band_label", min_rows=MIN_GROUP_VIDEOS, sort=False).sort_values("size_band_label")
video_annotator_table = summarize_groups(video_screen, "Annotator", min_rows=MIN_GROUP_VIDEOS)

display_and_save(video_size_table, "size_band_video", OUTPUT_DIR, n=10)
display_and_save(video_annotator_table, "annotator_video", OUTPUT_DIR, n=20)

frame_df.to_csv(OUTPUT_DIR / "frame_level_with_strata_and_failures.csv", index=False)
video_df.to_csv(OUTPUT_DIR / "video_level_with_strata_and_failures.csv", index=False)
print("Saved frame and video analysis surfaces.")

## 7. Case mix by size
How the size strata are distributed across annotator, surgery type, challenge, and resolution. Separates true model weakness from which cases each factor happened to get assigned.

In [ ]:
metadata_views = {
    "annotator_by_size": "Annotator",
    "surgery_by_size": "Surgery Type",
    "challenge_by_size": "challenge_primary",
    "resolution_by_size": "Video Resolution",
    "robotic_by_size": "Robotic",
}

for view_name, view_col in metadata_views.items():
    view_df = frame_df.copy()
    view_df[view_col] = collapse_rare(view_df[view_col], top_n=15, min_count=10)
    counts = pd.crosstab(view_df["size_band_label"], view_df[view_col], margins=True)
    rates = pd.crosstab(view_df["size_band_label"], view_df[view_col], values=view_df["failure_any"], aggfunc="mean").round(3)
    counts.to_csv(OUTPUT_DIR / f"{view_name}_counts.csv")
    rates.to_csv(OUTPUT_DIR / f"{view_name}_failure_rates.csv")
    print(f"\n{view_name}: frame counts")
    display(counts)
    print(f"{view_name}: failure rates")
    display(rates)

## 8. Case-mix-adjusted annotator weakness
Whether raw annotator differences from 01a hold up once case mix is accounted for.

In [ ]:
# Each frame is compared against the mean of frames with the same size, challenge, resolution
# and capture conditions, so an annotator who happened to get harder cases isn't penalised.
# Directional screen only, not a causal model.
adjusted = frame_df.copy()
adjusted["challenge_adjust"] = collapse_rare(adjusted["challenge_primary"], top_n=12, min_count=20)
adjusted["resolution_adjust"] = collapse_rare(adjusted["Video Resolution"], top_n=8, min_count=20)
adjusted["case_mix_stratum"] = (
    adjusted["size_band_label"].astype(str) + " | " +
    adjusted["challenge_adjust"].astype(str) + " | robotic=" + adjusted["Robotic"].astype(str) +
    " | no_tools=" + adjusted["No Tools"].astype(str) +
    " | res=" + adjusted["resolution_adjust"].astype(str)
)

# strata with fewer than 10 frames are pooled, otherwise a frame is compared against itself
stratum_counts = adjusted["case_mix_stratum"].value_counts()
adjusted["case_mix_stratum_pooled"] = adjusted["case_mix_stratum"].where(adjusted["case_mix_stratum"].map(stratum_counts).ge(10), "Sparse")
adjusted["expected_iou_case_mix"] = adjusted.groupby("case_mix_stratum_pooled")["iou"].transform("mean")
adjusted["expected_failure_case_mix"] = adjusted.groupby("case_mix_stratum_pooled")["failure_any"].transform("mean")
adjusted["iou_residual_case_mix"] = adjusted["iou"] - adjusted["expected_iou_case_mix"]
adjusted["failure_residual_case_mix"] = adjusted["failure_any"].astype(float) - adjusted["expected_failure_case_mix"]

annotator_adjusted = adjusted.groupby("Annotator", dropna=False).agg(
    n_frames=("key", "size"),
    n_videos=("Video ID", "nunique"),
    raw_iou_mean=("iou", "mean"),
    expected_iou_mean=("expected_iou_case_mix", "mean"),
    iou_residual_mean=("iou_residual_case_mix", "mean"),
    raw_failure_rate=("failure_any", "mean"),
    expected_failure_rate=("expected_failure_case_mix", "mean"),
    failure_residual_mean=("failure_residual_case_mix", "mean"),
    severe_iou_rate=("failure_severe_iou", "mean"),
    boundary_outlier_rate=("failure_boundary_outlier", "mean"),
    fp_outlier_rate=("failure_fp_outlier", "mean"),
).reset_index().sort_values(["failure_residual_mean", "iou_residual_mean"], ascending=[False, True])

print("Adjusted annotator screen: a positive failure residual means worse than comparable frames")
display_and_save(annotator_adjusted, "annotator_case_mix_adjusted", OUTPUT_DIR, n=20)

annotator_by_size = interaction_summary(adjusted, "Annotator", "size_band_label", min_rows=10)
annotator_by_challenge = interaction_summary(adjusted, "Annotator", "challenge_adjust", min_rows=10)

display_and_save(annotator_by_size, "annotator_by_size_failure", OUTPUT_DIR, n=40)
display_and_save(annotator_by_challenge, "annotator_by_challenge_failure", OUTPUT_DIR, n=40)

## 9. Interaction screens
Combinations of factors: size x challenge, thinness x size, size x annotator, surgery x challenge, and others. Ranked tables plus a challenge x size failure-rate heatmap, since the highest-yield failure surfaces are often combinations, not single factors.

In [ ]:
# tool-present frames only: no-tool frames trip almost every flag by construction and would
# dominate every ranking. They are screened separately in section 10.
interaction_df = frame_df[frame_df["has_tool_pixels"]].copy()
interaction_df["Surgery Type Top"] = collapse_rare(interaction_df["Surgery Type"], top_n=25, min_count=20)
interaction_df["Challenge Primary Top"] = collapse_rare(interaction_df["challenge_primary"], top_n=18, min_count=15)
interaction_df["Resolution Top"] = collapse_rare(interaction_df["Video Resolution"], top_n=10, min_count=10)

interaction_specs = [
    ("challenge_x_size", "Challenge Primary Top", "size_band_label"),
    ("thinness_x_size", "thin_cls_label", "size_band_label"),
    ("challenge_x_thinness", "Challenge Primary Top", "thin_cls_label"),
    ("annotator_x_size", "Annotator", "size_band_label"),
    ("surgery_x_size", "Surgery Type Top", "size_band_label"),
    ("surgery_x_challenge", "Surgery Type Top", "Challenge Primary Top"),
    ("resolution_x_size", "Resolution Top", "size_band_label"),
    ("robotic_x_size", "Robotic", "size_band_label"),
]

interaction_tables = {}
for interaction_name, left_col, right_col in interaction_specs:
    table = interaction_summary(interaction_df, left_col, right_col, min_rows=MIN_INTERACTION_FRAMES)
    interaction_tables[interaction_name] = table
    print(f"\n{interaction_name}")
    display_and_save(table, interaction_name, OUTPUT_DIR, n=30)

# compact view of the interaction the hypothesis cares about most
challenge_size_counts, challenge_size_rates = pivot_metric(interaction_df, "Challenge Primary Top", "size_band_label")
print("\nChallenge x size: frame counts")
display(challenge_size_counts)
print(f"Challenge x size: failure rate (cells under {MIN_INTERACTION_FRAMES} frames left blank)")
display(challenge_size_rates.round(3))

plot_heatmap_from_table(
    interaction_df,
    "Challenge Primary Top",
    "size_band_label",
    "failure_any",
    "Challenge x size failure rate",
    "challenge_x_size_heatmap.png",
    min_rows=MIN_INTERACTION_FRAMES,
)


## 10. Failure catalogue
Per-frame export of failure tags and metadata, for manual review of individual failing frames.

In [ ]:
catalog_cols = [
    "key", "image", "Video ID", "Timestamp", "Annotator", "Surgery Type", "Video Resolution", "Robotic", "No Tools",
    "Challenges", "challenge_primary", "size_band_label", "size_cls_label", "thin_cls_label", "width_px",
    "iou", "dice", "precision", "hd95", "msd", "asd", "tool_pixels", "fp_pixels", "tool_area_pct", "fp_area_pct",
    "failure_score", "failure_tags", *FAILURE_FLAG_COLS, "Notes",
]
catalog_cols = [col for col in catalog_cols if col in frame_df.columns]

failure_catalog = frame_df[frame_df["failure_any"]].copy()
failure_catalog = failure_catalog.sort_values(["failure_score", "iou", "hd95", "fp_pixels"], ascending=[False, True, False, False])
failure_catalog = failure_catalog[catalog_cols]

exports = {
    "failure_catalog": failure_catalog,
    "lowest_iou_frames": frame_df.sort_values("iou", ascending=True).head(50)[catalog_cols],
    "worst_boundary_frames": frame_df.sort_values("hd95", ascending=False).head(50)[catalog_cols],
    "highest_fp_frames": frame_df.sort_values("fp_pixels", ascending=False).head(50)[catalog_cols],
    "no_tool_false_positive_frames": frame_df[frame_df["failure_no_tool_fp"]].sort_values("fp_pixels", ascending=False)[catalog_cols],
}
for export_name, table in exports.items():
    table.to_csv(OUTPUT_DIR / f"{export_name}.csv", index=False)
    print(f"{export_name}: {len(table):,} rows -> {OUTPUT_DIR / (export_name + '.csv')}")

display(failure_catalog.head(40))

## 11. Directional effect screen
A combined linear screen ranking all listed factors at once, controlling for the others.

In [ ]:
# Least-squares fit on dummy-coded metadata. Ranks candidate predictors after accounting for
# the other listed factors, but it is not a causal model and does not replace section 13.
model_df = frame_df.copy()
model_df["Surgery Type Model"] = collapse_rare(model_df["Surgery Type"], top_n=20, min_count=25)
model_df["Challenge Model"] = collapse_rare(model_df["challenge_primary"], top_n=15, min_count=20)
model_df["Resolution Model"] = collapse_rare(model_df["Video Resolution"], top_n=8, min_count=20)
model_cols = ["Annotator", "Surgery Type Model", "Challenge Model", "Resolution Model", "Robotic", "No Tools", "size_band_label", "thin_cls_label"]

X = pd.get_dummies(model_df[model_cols].astype(str), drop_first=True, dtype=float)
X.insert(0, "intercept", 1.0)


def fit_lstsq_effects(target_col):
    y = pd.Series(pd.to_numeric(model_df[target_col], errors="coerce"), dtype=float)
    ok = y.notna() & np.isfinite(y)
    coef, *_ = np.linalg.lstsq(X.loc[ok].values, y.loc[ok].values, rcond=None)
    effects = pd.DataFrame({"feature": X.columns, f"{target_col}_coef": coef})
    effects = effects[effects["feature"].ne("intercept")].copy()
    effects["abs_coef"] = effects[f"{target_col}_coef"].abs()
    return effects.sort_values("abs_coef", ascending=False).drop(columns="abs_coef")


iou_effects = fit_lstsq_effects("iou")
failure_effects = fit_lstsq_effects("failure_any")

iou_effects.to_csv(OUTPUT_DIR / "directional_iou_effects.csv", index=False)
failure_effects.to_csv(OUTPUT_DIR / "directional_failure_effects.csv", index=False)

print("Most negative IoU coefficients are possible weak strata after controlling for other listed factors.")
display(iou_effects.sort_values("iou_coef").head(25))
print("Most positive failure coefficients are possible high-risk strata after controlling for other listed factors.")
display(failure_effects.sort_values("failure_any_coef", ascending=False).head(25))

## 12. Size, thinness and degradation figures
Headline charts for the hypothesis: mean IoU by tool size band and by thin/wide band, IoU under smoke, blood, motion blur and exposure, and a size-conditional panel that checks whether those conditions hurt every size band or mainly tiny tools.


In [ ]:
DEGRADATION_LABELS = ["Smoke", "Blood", "Motion Blur", "Underexposure", "Overexposure"]    # hypothesis degradation tokens
SIZE_BAND_ORDER = ["1_tiny", "2_small", "3_medium", "4_large"]

# == Size bands ==
size_band_table = summarize_groups(
    frame_df[frame_df["size_band"] > 0], "size_band_label", min_rows=MIN_GROUP_FRAMES, sort=False
).sort_values("size_band_label")
display_and_save(size_band_table, "size_band_frame", OUTPUT_DIR, n=10)


def plot_size_band_iou(size_table):
    labels = [pretty_size_label(value) for value in size_table["size_band_label"]]
    x = np.arange(len(size_table))
    yerr = np.vstack([
        size_table["iou_mean"] - size_table["iou_mean_ci_low"],
        size_table["iou_mean_ci_high"] - size_table["iou_mean"],
    ])
    colors = [COLORS["risk"] if label == "Tiny" else COLORS["volume"] for label in labels]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x, size_table["iou_mean"], yerr=yerr, capsize=5, color=colors, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Mean IoU")
    ax.set_xlabel("Tool size band (training cuts: <4.5%, <8%, <19%, >=19%)")
    ax.set_title("Segmentation quality by tool size")
    ax.grid(axis="y", alpha=0.25)
    for i, row in enumerate(size_table.itertuples()):
        ax.text(i, row.iou_mean + 0.05, f"{row.iou_mean:.2f}\nn={int(row.n_frames):,}", ha="center", fontsize=9)
    return save_and_show(fig, "iou_by_size_band.png")


plot_size_band_iou(size_band_table)

# == Thin/wide bands ==
# The run's own thin_class, cut at thin_cut_px, rather than a width recomputed here.
thin_band_table = summarize_groups(
    frame_df[frame_df["has_tool_pixels"]], "thin_cls_label", min_rows=MIN_GROUP_FRAMES, sort=False
).sort_values("thin_cls_label")
display_and_save(thin_band_table, "thin_band_frame", OUTPUT_DIR, n=10)


def plot_thin_band_iou(thin_table):
    labels = [pretty_size_label(value) for value in thin_table["thin_cls_label"]]
    x = np.arange(len(thin_table))
    yerr = np.vstack([
        thin_table["iou_mean"] - thin_table["iou_mean_ci_low"],
        thin_table["iou_mean_ci_high"] - thin_table["iou_mean"],
    ])
    colors = [COLORS["risk"] if label == "Thin" else COLORS["volume"] for label in labels]

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(x, thin_table["iou_mean"], yerr=yerr, capsize=5, color=colors, alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Mean IoU")
    ax.set_xlabel("Tool thinness band (mean local width, cut at the 25th percentile)")
    ax.set_title("Segmentation quality by tool thinness")
    ax.grid(axis="y", alpha=0.25)
    for i, row in enumerate(thin_table.itertuples()):
        ax.text(i, row.iou_mean + 0.05, f"{row.iou_mean:.2f}\nn={int(row.n_frames):,}", ha="center", fontsize=9)
    return save_and_show(fig, "iou_by_thin_band.png")


plot_thin_band_iou(thin_band_table)

# == Degraded visibility ==
# A frame can carry more than one challenge label, so these groups overlap by design.
tool_frames = frame_df[frame_df["has_tool_pixels"]]
degradation_groups = {"Normal": tool_frames[~tool_frames["has_challenge"]]}
for degradation_label in DEGRADATION_LABELS:
    degradation_groups[degradation_label] = tool_frames[tool_frames["challenge_tokens"].map(lambda xs: degradation_label in xs)]

degradation_table = pd.DataFrame([
    {
        "group": group_name,
        "n_frames": len(group),
        "n_videos": group["Video ID"].nunique(),
        "iou_mean": group["iou"].mean(),
        "iou_median": group["iou"].median(),
        "iou_std": group["iou"].std(),
        "iou_p10": group["iou"].quantile(0.10),
        "failure_any_rate": group["failure_any"].mean(),
    }
    for group_name, group in degradation_groups.items()
])
display_and_save(degradation_table, "degradation_frame", OUTPUT_DIR, n=10)


def plot_degradation_boxes(groups):
    values = [group["iou"].dropna().to_numpy() for group in groups.values()]
    labels = [f"{name}\nn={len(group):,}" for name, group in groups.items()]

    fig, ax = plt.subplots(figsize=(11, 5.5))
    box = ax.boxplot(
        values, showmeans=True, patch_artist=True, widths=0.55,
        whiskerprops={"color": COLORS["gray"], "linewidth": 1.2},
        capprops={"color": COLORS["gray"], "linewidth": 1.2},
        medianprops={"color": RULE, "linewidth": 1.6},
        flierprops={"marker": "o", "markersize": 3, "markerfacecolor": COLORS["gray"],
                    "markeredgecolor": "none", "alpha": 0.5},
    )
    for patch, name in zip(box["boxes"], groups):
        patch.set_facecolor(COLORS["good"] if name == "Normal" else COLORS["warning"])
        patch.set_alpha(0.75)
    ax.set_xticks(np.arange(1, len(values) + 1))
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel("IoU")
    ax.set_title("IoU distribution by frame condition (tool-present frames)")
    ax.grid(axis="y", alpha=0.25)
    return save_and_show(fig, "iou_by_degradation.png")


plot_degradation_boxes(degradation_groups)

# == Size-conditional degradation ==
# Same overlapping token rule as above, split by size band so a main-effect of tiny is not
# mistaken for a condition that hurts every size equally.
def degradation_x_size_rows(frames, size_order, degradation_labels):
    rows = []
    for size_label in size_order:
        size_frames = frames[frames["size_band_label"] == size_label]
        groups = {"Normal": size_frames[~size_frames["has_challenge"]]}
        for label in degradation_labels:
            groups[label] = size_frames[size_frames["challenge_tokens"].map(lambda xs, lab=label: lab in xs)]
        for group_name, group in groups.items():
            rows.append({
                "size_band_label": size_label,
                "group": group_name,
                "n_frames": len(group),
                "n_videos": group["Video ID"].nunique() if len(group) else 0,
                "iou_mean": group["iou"].mean() if len(group) else np.nan,
                "failure_any_rate": group["failure_any"].mean() if len(group) else np.nan,
            })
    return pd.DataFrame(rows)


degradation_x_size = degradation_x_size_rows(tool_frames, SIZE_BAND_ORDER, DEGRADATION_LABELS)
display_and_save(degradation_x_size, "degradation_x_size", OUTPUT_DIR, n=40)


def plot_iou_by_size_and_degradation(table, size_order, degradation_labels, min_rows=MIN_INTERACTION_FRAMES):
    conditions = ["Normal", *degradation_labels]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
    for ax, size_label in zip(axes.ravel(), size_order):
        subset = table[table["size_band_label"] == size_label].set_index("group")
        heights = []
        colors = []
        annotations = []
        for condition in conditions:
            if condition not in subset.index:
                heights.append(np.nan)
                colors.append(COLORS["gray"])
                annotations.append("n=0")
                continue
            row = subset.loc[condition]
            n_frames = int(row["n_frames"])
            if n_frames < min_rows:
                heights.append(np.nan)
                colors.append(COLORS["gray"])
                annotations.append(f"n={n_frames}")
                continue
            heights.append(float(row["iou_mean"]))
            colors.append(COLORS["good"] if condition == "Normal" else COLORS["warning"])
            annotations.append(f"n={n_frames}")
        x = np.arange(len(conditions))
        bars = ax.bar(x, [0 if np.isnan(h) else h for h in heights], color=colors, alpha=0.9)
        for bar, height, note in zip(bars, heights, annotations):
            if np.isnan(height):
                bar.set_alpha(0.15)
                ax.text(bar.get_x() + bar.get_width() / 2, 0.05, note, ha="center", va="bottom", fontsize=7, color=COLORS["gray"], rotation=90)
            else:
                ax.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f"{height:.2f}\n{note}", ha="center", va="bottom", fontsize=7)
        ax.set_xticks(x)
        ax.set_xticklabels(conditions, rotation=35, ha="right", fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_title(pretty_size_label(size_label))
        ax.grid(axis="y", alpha=0.25)
        if size_label in ("1_tiny", "3_medium"):
            ax.set_ylabel("Mean IoU")
    fig.suptitle(f"Mean IoU by size band and degradation (cells under {min_rows} frames blanked)", y=1.01)
    return save_and_show(fig, "iou_by_size_and_degradation.png")


plot_iou_by_size_and_degradation(degradation_x_size, SIZE_BAND_ORDER, DEGRADATION_LABELS)


## 13. Worst 20 galleries
The 20 lowest-IoU frames are shown next to their ground truth and prediction masks, so missed tools and over-segmented tissue can be checked by eye rather than inferred from the tables.

Needs `FRAME_DATA_ROOT` and `PRED_MASKS_ROOT` from the setup cell (synced from GCS).

In [ ]:
from PIL import Image
from matplotlib.patches import Patch

WORST_N = 20
ROWS_PER_FIGURE = 5          # one figure per 5 frames, otherwise the output is one huge image
DISPLAY_SIZE = (512, 512)    # masks were scored at 512 x 512, so show everything at that size

CONFUSION_KEY = [
    Patch(facecolor="green", label="TP, correct"),
    Patch(facecolor="red", label="FP, over-segmented"),
    Patch(facecolor="blue", label="FN, missed"),
]


def gallery_paths(image_path):
    """Map a CSV Colab path to local (rgb, gt, pred) under FRAME_DATA_ROOT / PRED_MASKS_ROOT."""
    parts = Path(str(image_path)).parts
    if "JPEGImages" in parts:
        rel = Path(*parts[parts.index("JPEGImages") + 1:])
        stem = rel.with_suffix(".png")
        return (
            FRAME_DATA_ROOT / "JPEGImages" / rel,
            FRAME_DATA_ROOT / "SegmentationClass" / stem,
            PRED_MASKS_ROOT / "JPEGImages" / stem,
        )
    if "NoTools" in parts:
        rel = Path(*parts[parts.index("NoTools") + 1:])
        stem = rel.with_suffix(".png")
        return (
            FRAME_DATA_ROOT / "NoTools" / rel,
            None,
            PRED_MASKS_ROOT / "NoTools" / stem,
        )
    name = Path(str(image_path)).name
    return (
        IMAGES_DIR / name,
        MASKS_DIR / Path(name).with_suffix(".png"),
        PRED_MASKS_ROOT / "JPEGImages" / Path(name).with_suffix(".png"),
    )


def load_rgb(path):
    if path is None or not Path(path).exists():
        return None
    return np.asarray(Image.open(path).convert("RGB").resize(DISPLAY_SIZE, Image.Resampling.BILINEAR))


def load_binary_mask(path):
    # masks are RGB with a single foreground colour, so any non-black pixel is an instrument
    if path is None or not Path(path).exists():
        return None
    mask = Image.open(path).convert("RGB").resize(DISPLAY_SIZE, Image.Resampling.NEAREST)
    return np.asarray(np.asarray(mask).max(axis=-1) > 0, dtype=np.uint8)


def confusion_rgb(pred, gt):
    out = np.zeros((*gt.shape, 3))
    out[(pred == 1) & (gt == 1)] = [0, 1, 0]
    out[(pred == 1) & (gt == 0)] = [1, 0, 0]
    out[(pred == 0) & (gt == 1)] = [0, 0, 1]
    return out


def show_panel(ax: Axes, data, cmap=None, missing="file not found"):
    if data is None:
        ax.text(0.5, 0.5, missing, ha="center", va="center", fontsize=8, color=COLORS["gray"])
    else:
        ax.imshow(data, cmap=cmap)
    ax.set_xticks([])
    ax.set_yticks([])


def show_worst_gallery(frames):
    columns = ["Input frame", "Ground truth", "Prediction", "TP(g) / FP(r) / FN(b)"]
    for page_start in range(0, len(frames), ROWS_PER_FIGURE):
        page = frames.iloc[page_start:page_start + ROWS_PER_FIGURE]
        fig, axes = plt.subplots(len(page), 4, figsize=(13, 3.4 * len(page)))
        axes = np.atleast_2d(axes)
        for col_i, column_title in enumerate(columns):
            ax_at(axes, 0, col_i).set_title(column_title, fontsize=10)

        for row_i, (_, frame) in enumerate(page.iterrows()):
            image_path, gt_path, pred_path = gallery_paths(frame["image"])
            image = load_rgb(image_path)
            gt = load_binary_mask(gt_path)
            pred = load_binary_mask(pred_path)

            show_panel(ax_at(axes, row_i, 0), image)
            show_panel(ax_at(axes, row_i, 1), gt, cmap="gray")
            show_panel(ax_at(axes, row_i, 2), pred, cmap="gray", missing="prediction not exported")
            show_panel(ax_at(axes, row_i, 3), confusion_rgb(pred, gt) if gt is not None and pred is not None else None)
            ax_at(axes, row_i, 0).set_ylabel(
                f"{frame['key']}\nIoU={frame['iou']:.3f}  area={frame['tool_area_pct']:.2f}%\n{frame['size_band_label']}",
                fontsize=8,
            )
        # keep the key above the grid so it does not sit on the confusion panel
        fig.legend(
            handles=CONFUSION_KEY,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.0),
            ncol=3,
            fontsize=8,
            frameon=False,
        )
        fig.suptitle(f"Lowest-IoU frames {page_start + 1} to {page_start + len(page)} of {len(frames)}", fontsize=12, y=1.06)
        save_and_show(fig, f"worst_{WORST_N}_gallery_{page_start // ROWS_PER_FIGURE + 1}.png")


# no-tool frames are excluded: IoU against an empty ground truth is not comparable
worst_frames = frame_df[frame_df["has_tool_pixels"]].nsmallest(WORST_N, "iou").reset_index(drop=True)
worst_frames[catalog_cols].to_csv(OUTPUT_DIR / f"worst_{WORST_N}_frames.csv", index=False)
display(worst_frames[["key", "Video ID", "iou", "precision", "tool_area_pct", "fp_area_pct", "size_band_label", "Challenges", "failure_tags"]])

missing_img = sum(1 for p in worst_frames["image"] if not gallery_paths(p)[0].exists())
missing_gt = sum(1 for p in worst_frames["image"] if (gp := gallery_paths(p)[1]) is not None and not gp.exists())
missing_pred = sum(1 for p in worst_frames["image"] if not gallery_paths(p)[2].exists())
print(f"Gallery files missing among worst {WORST_N}: image={missing_img}  gt={missing_gt}  pred={missing_pred}")

show_worst_gallery(worst_frames)


## 14. Finding scaffold
Prints the highest-priority leads to verify manually in the exported failure catalogue.

In [ ]:
lead_cols = ["n_frames", "n_videos", "iou_mean", "delta_iou_vs_global", "failure_any_rate", "delta_failure_vs_global"]

HYPOTHESIS_CHALLENGES = ["Smoke", "Blood", "Underexposure", "Overexposure", "Motion Blur"]

print("1. Tool size bands, frame level")
display(size_band_table[["size_band_label", *lead_cols, "severe_iou_rate"]])

print("1a. Tool thinness bands, frame level")
display(thin_band_table[["thin_cls_label", *lead_cols, "severe_iou_rate"]])

print("2. Tool size bands, video level")
display(video_size_table[["size_band_label", *lead_cols]])

print("3. Degraded visibility groups")
display(degradation_table)

print("4. Annotator leads after case-mix adjustment")
display(annotator_adjusted.head(10))

print("5. Highest-risk interactions")
all_interactions = []
for interaction_name, table in interaction_tables.items():
    tagged = table.copy()
    tagged.insert(0, "interaction", interaction_name)
    all_interactions.append(tagged)
all_interactions = pd.concat(all_interactions, ignore_index=True) if all_interactions else pd.DataFrame()
if len(all_interactions):
    all_interactions = all_interactions.sort_values(["failure_any_rate", "iou_mean", "n_frames"], ascending=[False, True, False])
    all_interactions.to_csv(OUTPUT_DIR / "top_interaction_leads.csv", index=False)
    display(all_interactions.head(30))

print("6. Hypothesis challenge x size cells (augmentation targets)")
# primary-challenge x size from section 9, restricted to the degradation tokens in the hypothesis
challenge_x_size = interaction_tables.get("challenge_x_size", pd.DataFrame())
if len(challenge_x_size):
    challenge_col = "Challenge Primary Top"
    hyp_cells = challenge_x_size[challenge_x_size[challenge_col].isin(HYPOTHESIS_CHALLENGES)].copy()
    hyp_cells = hyp_cells.sort_values(["failure_any_rate", "iou_mean", "n_frames"], ascending=[False, True, False])
    hyp_cols = [challenge_col, "size_band_label", "n_frames", "n_videos", "iou_mean", "failure_any_rate", "delta_iou_vs_global", "delta_failure_vs_global"]
    hyp_cols = [c for c in hyp_cols if c in hyp_cells.columns]
    display_and_save(hyp_cells[hyp_cols], "hypothesis_challenge_x_size", OUTPUT_DIR, n=30)
else:
    print("challenge_x_size table missing; re-run section 9")

print("7. Frames reviewed by eye in section 13")
display(worst_frames[["key", "Video ID", "iou", "tool_area_pct", "Challenges", "failure_tags"]])

print("\nTables and exports:", OUTPUT_DIR)
print("Figures:", VISUAL_DIR)